# fine-tune-forge — QLoRA Fine-tuning Pipeline

Pipeline per il fine-tuning di piccoli LLM su task verticali.

**Prima di eseguire**: `Runtime → Change runtime type → T4 GPU`

**Output persistenti**:
- Dataset JSONL → scaricabile da Colab o rigenerabile (pochi MB)
- Modello fine-tunato → **HuggingFace Hub** (unico storage pratico per GB di pesi)

## 1. Setup

In [ ]:
# Controlla GPU disponibile
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✓ GPU: {result.stdout.strip()}')
else:
    print('⚠️  Nessuna GPU — vai su Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clona il repo e installa dipendenze
!git clone -q https://github.com/MatteoAdamo82/fine-tuning.git /content/fine-tuning
%cd /content/fine-tuning
!pip install -q -e . 2>&1 | tail -3
print('✓ Installazione completata')

## 2. Configurazione

Aggiungi le API key in **Secrets** (icona 🔑 nella barra sinistra):
- `GEMINI_API_KEY` — da [aistudio.google.com](https://aistudio.google.com)
- `HF_TOKEN` — da [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (write access)

In [ ]:
import os
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('✓ API keys caricate')

In [ ]:
# ── Parametri del run — modifica qui ──────────────────────────────────────────
DOMINIO  = 'restaurant_booking'   # restaurant_booking | sales_agent | professional_booking
MODELLO  = 'Qwen/Qwen3-0.6B'     # Qwen/Qwen3-0.6B (~2GB VRAM) | Qwen/Qwen3-4B (~6GB)
N_ESEMPI = 50                     # 50 per test rapido, 150 per produzione
HF_REPO  = ''                     # es. 'MatteoAdamo82/agente-ristorante' — obbligatorio per salvare
# ──────────────────────────────────────────────────────────────────────────────

if not HF_REPO:
    print('⚠️  HF_REPO non impostato: il modello non verrà salvato dopo il training')
else:
    print(f'✓ Il modello verrà pushato su: https://huggingface.co/{HF_REPO}')
print(f'Dominio: {DOMINIO} | Modello: {MODELLO} | Esempi: {N_ESEMPI}')

## 3. Verifica installazione

In [ ]:
!forge list-domains

## 4. Generazione Dataset (Gemini)

In [ ]:
DATASET_PATH = f'data/processed/{DOMINIO}.jsonl'

!forge dataset \
    --dominio {DOMINIO} \
    --n-examples {N_ESEMPI} \
    --output {DATASET_PATH}

In [ ]:
# Anteprima dataset
import json
with open(DATASET_PATH) as f:
    esempi = [json.loads(l) for l in f if l.strip()]
print(f'Esempi generati: {len(esempi)}')
print('\n--- Primo esempio ---')
for msg in esempi[0]['messages']:
    print(f"[{msg['role']}] {msg['content'][:150]}")

## 5. Training QLoRA

In [ ]:
import uuid
RUN_ID = f'{DOMINIO}-{uuid.uuid4().hex[:6]}'
print(f'Run ID: {RUN_ID}')

!forge train \
    {DATASET_PATH} \
    --dominio {DOMINIO} \
    --modello {MODELLO} \
    --run-id {RUN_ID}

In [ ]:
from pathlib import Path
merged_path = Path(f'outputs/checkpoints/{RUN_ID}/merged')
if merged_path.exists():
    print(f'✓ Modello merged: {merged_path}')
    print(f'  File: {[f.name for f in merged_path.iterdir()]}')
else:
    raise FileNotFoundError(f'Merged model non trovato: {merged_path}')

## 6. Export su HuggingFace Hub

Il modello (1-2GB) viene pushato su HF Hub — unico storage pratico per pesi di modelli.
Il dataset è piccolo e rigenerabile con Gemini in qualsiasi momento.

In [ ]:
if HF_REPO:
    !forge export {merged_path} --format hf --repo-id {HF_REPO}
    print(f'\n✓ Modello disponibile su: https://huggingface.co/{HF_REPO}')
else:
    print('⚠️  HF_REPO non impostato — imposta la variabile nella cella di configurazione e riesegui')

## 7. Test del modello

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, yaml

tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
model = AutoModelForCausalLM.from_pretrained(
    str(merged_path),
    torch_dtype=torch.float16,
    device_map='auto'
)

with open(f'config/domains/{DOMINIO}.yaml') as f:
    cfg = yaml.safe_load(f)
system_prompt = cfg.get('dataset', {}).get('obiettivo', 'Sei un assistente utile.')

def chat(user_msg):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_msg}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

risposta = chat('Vorrei prenotare un tavolo per sabato sera per 4 persone.')
print(f'User: Vorrei prenotare un tavolo per sabato sera per 4 persone.')
print(f'\nAssistant: {risposta}')